# Conteo de Personas Cenital con CSRNet — Mapa de Densidad
Detecta y cuenta personas en videos cenitales generando un **mapa de calor de densidad**.
Funciona bien tanto con pocas como con muchas personas juntas.

## 1. Instalación de dependencias

In [2]:
!pip install torch torchvision opencv-python-headless tqdm scipy Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 44.7 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 60.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 47.9 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 58.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 49.6 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 55.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 57.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 41.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 54.8 

## 2. Importar librerías

In [4]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from tqdm import tqdm
from PIL import Image
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Librerías cargadas  Dispositivo: {DEVICE}')

Librerías cargadas | Dispositivo: cpu


## 3. Arquitectura CSRNet

In [5]:
class CSRNet(nn.Module):
    """CSRNet (CVPR 2018): Frontend VGG-16 + Backend de convoluciones dilatadas.
    Genera un mapa de densidad donde la suma = número estimado de personas."""

    def __init__(self):
        super().__init__()
        self.frontend = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),   nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128,128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128,256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256,256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256,256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256,512, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512,512, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512,512, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.backend = nn.Sequential(
            nn.Conv2d(512,512,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512,512,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512,512,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512,256,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256,128,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(64,   1,1),
        )

    def forward(self, x):
        return self.backend(self.frontend(x))

print(' Arquitectura CSRNet definida')

✅ Arquitectura CSRNet definida


## 4. Configuración — **edita aquí**

In [6]:
# ─── CAMBIA ESTOS VALORES ────────────────────────────────────────────
VIDEO_PATH   = 'data/videos/videoTM_02.mkv'          # Ruta a tu video ceni/tal
OUTPUT_VIDEO = 'outputs/videos/output_density.mp4'    # Video anotado con heatmap
OUTPUT_CSV   = 'conteo_density.csv'    # CSV con conteo por frame
WEIGHTS_PATH = 'csrnet_partA.pth'      # Checkpoint CSRNet (opcional, ver celda 5)

SKIP_FRAMES   = 2       # Procesar 1 de cada N frames (2 = mitad de frames)
ALPHA_OVERLAY = 0.55    # Opacidad del heatmap sobre el video (0.0–1.0)
COLORMAP      = cm.jet  # Paleta: cm.jet | cm.hot | cm.plasma | cm.inferno
# ─────────────────────────────────────────────────────────────────────

assert os.path.exists(VIDEO_PATH), f' Video no encontrado: {VIDEO_PATH}'
print(f' Video encontrado: {VIDEO_PATH}')

 Video encontrado: data/videos/videoTM_02.mkv


## 5. Cargar modelo
> Si tienes un checkpoint `.pth` pre-entrenado en ShanghaiTech o VisDrone, ponlo en `WEIGHTS_PATH`.
> Si no, se cargan los pesos VGG-16 de ImageNet como punto de partida (frontend).

In [7]:
model = CSRNet().to(DEVICE)

if os.path.exists(WEIGHTS_PATH):
    ckpt  = torch.load(WEIGHTS_PATH, map_location=DEVICE)
    state = ckpt.get('state_dict', ckpt)
    model.load_state_dict(state, strict=False)
    print(f' Checkpoint CSRNet cargado: {WEIGHTS_PATH}')
else:
    print(f' Checkpoint no encontrado. Cargando VGG-16 ImageNet en el frontend...')
    vgg        = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    vgg_convs  = [l for l in vgg.features if isinstance(l, nn.Conv2d)]
    csr_convs  = [l for l in model.frontend if isinstance(l, nn.Conv2d)]
    for dst, src in zip(csr_convs, vgg_convs[:len(csr_convs)]):
        dst.weight.data.copy_(src.weight.data)
        dst.bias.data.copy_(src.bias.data)
    print('   Pesos VGG-16 copiados al frontend')
    print('   Para mayor precisión entrena con ShanghaiTech o VisDrone.')

model.eval()
print(' Modelo en modo evaluación')

⚠️  Checkpoint no encontrado. Cargando VGG-16 ImageNet en el frontend...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:12<00:00, 45.2MB/s] 


   ✅ Pesos VGG-16 copiados al frontend
   💡 Para mayor precisión entrena con ShanghaiTech o VisDrone.
✅ Modelo en modo evaluación


## 6. Funciones auxiliares

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

def preprocess(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return transform(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)

def overlay_heatmap(bgr, density, alpha=ALPHA_OVERLAY):
    h, w  = bgr.shape[:2]
    d_min, d_max = density.min(), density.max()
    norm  = (density - d_min) / (d_max - d_min + 1e-8)
    heat  = (COLORMAP(norm)[:,:,:3] * 255).astype(np.uint8)
    heat  = cv2.cvtColor(heat, cv2.COLOR_RGB2BGR)
    heat  = cv2.resize(heat, (w, h), interpolation=cv2.INTER_LINEAR)
    return cv2.addWeighted(bgr, 1 - alpha, heat, alpha, 0)

print('Funciones listas')

✅ Funciones listas


## 7. Procesar video y generar salida

In [9]:
cap   = cv2.VideoCapture(VIDEO_PATH)
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS   = cap.get(cv2.CAP_PROP_FPS)
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'{W}x{H} @ {FPS:.1f} fps | {TOTAL} frames')

writer = cv2.VideoWriter(
    OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H)
)

records, frame_idx = [], 0
last_density, last_count = None, 0.0

with torch.no_grad():
    with tqdm(total=TOTAL, desc='Procesando') as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % SKIP_FRAMES == 0:
                density      = model(preprocess(frame)).squeeze().cpu().numpy()
                last_density = density
                last_count   = float(density.sum())
                records.append({
                    'frame'    : frame_idx,
                    'timestamp': round(frame_idx / FPS, 3),
                    'personas' : round(last_count, 2)
                })

            out = overlay_heatmap(frame, last_density) if last_density is not None else frame.copy()

            # HUD
            cv2.rectangle(out, (10,10), (260,52), (0,0,0), -1)
            cv2.putText(out, f'Personas: {last_count:.1f}',
                        (15,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)

            writer.write(out)
            frame_idx += 1
            pbar.update(1)

cap.release(); writer.release()
print(f'\n Video guardado: {OUTPUT_VIDEO}')

📹 640x480 @ 30.0 fps | 26999 frames


Procesando: 100%|██████████| 26999/26999 [2:01:04<00:00,  3.72it/s]  


✅ Video guardado: outputs/videos/output_density.mp4


## 8. CSV y estadísticas

In [ ]:
df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)
print(f' CSV guardado: {OUTPUT_CSV}\n')
print(f'  Máximo  : {df.personas.max():.1f} personas')
print(f'  Promedio: {df.personas.mean():.2f} personas')
print(f'  Mínimo  : {df.personas.min():.1f} personas')
df.tail(6)

## 9. Gráfica de conteo en el tiempo

In [ ]:
fig, ax = plt.subplots(figsize=(14,4))
ax.fill_between(df.timestamp, df.personas, alpha=0.2, color='darkorange')
ax.plot(df.timestamp, df.personas, color='darkorange', lw=1.5, label='Personas estimadas')
ax.axhline(df.personas.mean(), color='steelblue', ls='--', lw=1.2,
           label=f'Promedio ({df.personas.mean():.1f})')
ax.set(xlabel='Tiempo (s)', ylabel='Personas (estimado)',
       title='Conteo cenital — Mapa de densidad CSRNet')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('conteo_density_plot.png', dpi=150)
plt.show()

## 10. Visualizar frame pico + mapa de densidad

In [ ]:
peak_idx = int(df.loc[df.personas.idxmax(), 'frame'])
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, peak_idx)
ret, frame = cap.read(); cap.release()

if ret:
    with torch.no_grad():
        density = model(preprocess(frame)).squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14,6))
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Frame cenital original', fontsize=13)
    axes[0].axis('off')

    im = axes[1].imshow(density, cmap='jet')
    axes[1].set_title(f'Mapa de densidad — {density.sum():.1f} personas', fontsize=13)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.suptitle(f'Frame #{peak_idx} — pico de afluencia', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('density_map_peak.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Imagen guardada: density_map_peak.png')

---
## 💡 Ajustes para mejorar precisión

| Situación | Solución |
|---|---|
| Conteo muy alto o bajo | Calibrar: multiplica `density.sum()` por un factor contra una muestra manual |
| Heatmap borroso | Reducir `ALPHA_OVERLAY` o aplicar `gaussian_filter(density, sigma=1)` |
| Lento en CPU | Subir `SKIP_FRAMES` a 3–5 y bajar resolución del video con `cv2.resize` |
| Quiero más precisión cenital | Fine-tunear con **VisDrone** o **DroneCrowd** datasets |
| Fine-tuning | Repositorio oficial: https://github.com/leeyeehoo/CSRNet-pytorch |